# Wildfire Risk Prediction
## AlphaEarth Foundation Embeddings + NASA FIRMS

This notebook walks through:
1. **Data loading** — synthetic dev data (swap with GEE calls for production)
2. **EDA** — embedding distributions, PCA, fire/non-fire separation
3. **Temporal analysis** — how embeddings drift as landscape dries
4. **Model training** — XGBoost + LightGBM ensemble
5. **SHAP explainability** — what drives predictions
6. **Risk mapping** — spatial visualization of predicted probabilities

## 1. Setup

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA

from src.data_loader import load_sample_data_for_dev
from src.features import WildfireFeatureBuilder
from src.model import WildfireEnsemble, evaluate_model, WildfireSHAPExplainer
from src.visualize import (
    plot_risk_map, plot_evaluation_suite,
    plot_shap_summary, plot_feature_group_contributions
)

plt.rcParams["figure.dpi"] = 120
print("Setup complete")


## 2. Load Data

In [ ]:
# Load synthetic dev data
# Replace with GEE calls in production (see src/data_loader.py)
df = load_sample_data_for_dev(n_fire=8000, n_nonfire=80000, seed=42)

print(f"Dataset: {len(df):,} samples")
print(f"Fire pixels: {df.label.sum():,} ({df.label.mean()*100:.1f}%)")
print(f"Columns: {len(df.columns)}")
df.head(3)


## 3. Exploratory Data Analysis
### 3.1 Embedding Distribution

In [ ]:
emb_cols = [f"embedding_{i}" for i in range(64)]

fig, axes = plt.subplots(2, 4, figsize=(16, 7))
axes = axes.ravel()

# Show first 8 embedding dimensions, split by label
for i in range(8):
    col = f"embedding_{i}"
    fire = df[df.label == 1][col]
    nofire = df[df.label == 0][col]
    axes[i].hist(nofire.sample(5000), bins=50, alpha=0.6, label="No fire", color="#4caf50", density=True)
    axes[i].hist(fire.sample(min(5000,len(fire))), bins=50, alpha=0.6, label="Fire", color="#f44336", density=True)
    axes[i].set_title(f"Dim {i}")
    if i == 0: axes[i].legend()

plt.suptitle("AEF Embedding Distributions: Fire vs Non-Fire Pixels", fontsize=13)
plt.tight_layout()
plt.show()


### 3.2 PCA — Foundation Model Embedding Space

In [ ]:
emb_data = df[emb_cols].values
pca = PCA(n_components=2, random_state=42)
pca_proj = pca.fit_transform(emb_data)

fig, ax = plt.subplots(figsize=(10, 7))
nofire_mask = df.label == 0
fire_mask = df.label == 1

ax.scatter(pca_proj[nofire_mask, 0], pca_proj[nofire_mask, 1],
           c="#4caf50", s=0.5, alpha=0.3, label=f"No fire ({nofire_mask.sum():,})", rasterized=True)
ax.scatter(pca_proj[fire_mask, 0], pca_proj[fire_mask, 1],
           c="#f44336", s=3, alpha=0.8, label=f"Fire ({fire_mask.sum():,})", rasterized=True)

ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)")
ax.set_title("AEF Embedding Space (PCA)
Fire pixels cluster in a distinct region of latent space")
ax.legend(markerscale=6)
plt.tight_layout()
plt.show()

print(f"PC1+PC2 explain {pca.explained_variance_ratio_[:2].sum()*100:.1f}% of variance")


### 3.3 Temporal Delta Analysis

In [ ]:
delta_cols = [f"delta_{i}" for i in range(64)]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Delta norm distribution
delta_norm = np.linalg.norm(df[delta_cols].values, axis=1)
for label_val, color, name in [(0, "#4caf50", "No fire"), (1, "#f44336", "Fire")]:
    mask = df.label == label_val
    axes[0].hist(delta_norm[mask], bins=60, alpha=0.6, color=color, label=name, density=True)
axes[0].set_title("Embedding Drift Magnitude")
axes[0].set_xlabel("||Δembedding||")
axes[0].legend()

# Mean delta (positive = moving toward drier signature)
delta_mean = df[delta_cols].values.mean(axis=1)
for label_val, color, name in [(0, "#4caf50", "No fire"), (1, "#f44336", "Fire")]:
    mask = df.label == label_val
    axes[1].hist(delta_mean[mask], bins=60, alpha=0.6, color=color, label=name, density=True)
axes[1].set_title("Drift Direction (+ = drying)")
axes[1].set_xlabel("Mean Δembedding")
axes[1].axvline(0, color="black", lw=1, linestyle="--")
axes[1].legend()

# Fraction of dims increasing (drying indicator)
pos_frac = (df[delta_cols].values > 0).mean(axis=1)
for label_val, color, name in [(0, "#4caf50", "No fire"), (1, "#f44336", "Fire")]:
    mask = df.label == label_val
    axes[2].hist(pos_frac[mask], bins=30, alpha=0.6, color=color, label=name, density=True)
axes[2].set_title("Fraction of Dims Increasing")
axes[2].set_xlabel("Fraction")
axes[2].axvline(0.5, color="black", lw=1, linestyle="--")
axes[2].legend()

plt.suptitle("Temporal Embedding Drift: Fire vs Non-Fire", fontsize=13)
plt.tight_layout()
plt.show()


## 4. Feature Engineering

In [ ]:
builder = WildfireFeatureBuilder(
    use_temporal_delta=True,
    use_spatial_lag=True,
    use_embedding_norm=True,
)

X, feature_names = builder.fit_transform(df)
y = df["label"].values

print(f"Feature matrix: {X.shape[0]:,} × {X.shape[1]}")
print(f"Feature groups:")
print(f"  Embedding dims:   {sum(1 for f in feature_names if f.startswith("emb_")):>4}")
print(f"  Delta dims:       {sum(1 for f in feature_names if f.startswith("delta_")):>4}")
print(f"  Spatial lag:      {sum(1 for f in feature_names if f.startswith("spatial_")):>4}")
print(f"  Fire history:     {sum(1 for f in feature_names if f.startswith("fire_") or f.startswith("years_")):>4}")
print(f"  Topography:       {sum(1 for f in feature_names if f in ["elevation","slope","aspect"]):>4}")
print(f"  Summary stats:    {sum(1 for f in feature_names if f.startswith("emb_") and not f.replace("emb_","").isdigit()):>4}")


## 5. Train/Test Split & Model Training

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {len(X_train):,} | Test: {len(X_test):,}")
print(f"Train fire rate: {y_train.mean():.3f} | Test fire rate: {y_test.mean():.3f}")


In [ ]:
# Train ensemble
# NOTE: For production, use spatial_cross_validate() from src/model.py
model = WildfireEnsemble()
model.fit(X_train, y_train, X_test, y_test, feature_names=feature_names)


## 6. Evaluation

In [ ]:
metrics = evaluate_model(model, X_test, y_test, split_name="test")


In [ ]:
y_prob = model.predict_proba(X_test)[:, 1]
fig = plot_evaluation_suite(y_test, y_prob)
plt.show()


## 7. SHAP Explainability

In [ ]:
explainer = WildfireSHAPExplainer(model, feature_names)
shap_values, X_sample = explainer.compute_shap_values(X_test, max_samples=2000)

print("Top 15 features by SHAP importance:")
print(explainer.top_features(shap_values, n=15).to_string(index=False))


In [ ]:
# Feature group contributions
group_contrib = explainer.embedding_interpretation(shap_values)
fig = plot_feature_group_contributions(group_contrib)
plt.show()
print(group_contrib.to_string(index=False))


## 8. Risk Map

In [ ]:
# Add predictions to test set
import numpy as np
idx_test = np.arange(len(X_test))
df_test = df.iloc[-len(X_test):].copy()
df_test["fire_prob"] = model.predict_proba(X_test)[:, 1]
df_test["risk_tier"] = model.predict_risk_tier(X_test)

print("Risk tier breakdown:")
print(df_test["risk_tier"].value_counts())


In [ ]:
fig = plot_risk_map(
    df_test,
    title="Wildfire Risk Map — 2024 Predictions",
    figsize=(12, 8)
)
plt.show()


## 9. Next Steps

**To use real data:**
1. Run  in terminal
2. Set your GEE project in 
3. Get a free NASA FIRMS API key at https://firms.modaps.eosdis.nasa.gov/api/
4. Run 

**To extend the model:**
- Try a GNN (graph neural network) where nodes = pixels and edges = spatial adjacency
- Add real-time FIRMS streaming for daily risk updates
- Fine-tune on a specific fire season to improve recall for extreme events
- Add uncertainty quantification with conformal prediction intervals